# Annotate a corpus with the tuned ModernBERT fact-span tagger

Runs the fine-tuned **ModernBERT-large** span tagger over a corpus stored in Google Drive and
writes fact-span annotations back to Drive. This is the *fact-identification* half of the Co-LMLM
annotation pipeline — it tags where the facts are (BIO), producing verbatim spans with character
offsets. No question generator is run.

| | |
|---|---|
| Input | `edullm/fineweb-edu-1b-smollm2-raw/shards/*.jsonl.gz` under Drive folder `14l8nIqnNDXIl7ZCEM6qu7tRJPkSJy0L5` |
| Model | your tuned `ModernBertForTokenClassification` checkpoint in Drive |
| Output | sharded `*.annotations.jsonl.zst` in your My Drive |
| Hardware | any GPU (A100 ideal; T4 fine — the tagger is small) |

**How access works.** A folder *shared with your account* is **not** visible under a mounted Drive
(`drive.mount` only exposes *My Drive*, not "Shared with me"). So the input is read through the
**Drive API** using Colab's own auth, which reaches the folder by its ID. Output is written to your
mounted My Drive, which is writable.

**Resumable.** Each input file is annotated to its own output shard and recorded in a manifest, so
if the session drops you just re-run the notebook and it skips finished files. **Test first** by
setting `MAX_FILES = 1` before committing to the whole corpus.

## 1. Runtime

Use a GPU runtime (**Runtime → Change runtime type → GPU**). The tagger runs on CPU too, but far
slower.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU — tagging will be slow. Runtime -> Change runtime type -> GPU.")

In [ ]:
%pip install -q -U "transformers>=4.48" zstandard pyarrow datasets

## 2. Authenticate + mount

Two auth steps: `authenticate_user()` lets the Drive API read the shared input folder; `drive.mount`
gives a writable path to your My Drive for the output.

In [ ]:
from google.colab import auth, drive

auth.authenticate_user()      # for the Drive API (reads the shared input folder by ID)
drive.mount("/content/drive")  # for writing output to My Drive

In [ ]:
from pathlib import Path

# ---- where the documents come from ----
# "auto"  : try the Drive folder; if it holds no text documents (e.g. a pre-tokenized
#           corpus), stream the upstream HF dataset named in its meta.json
# "drive" : only read the Drive folder
# "hf"    : only stream the HF dataset configured below
INPUT_MODE = "drive"

# Root of the shared "edullm" Drive folder. Walked recursively, so subfolders
# (individual datasets) are picked up without naming them here.
# https://drive.google.com/drive/folders/14l8nIqnNDXIl7ZCEM6qu7tRJPkSJy0L5
INPUT_FOLDER_ID = "14l8nIqnNDXIl7ZCEM6qu7tRJPkSJy0L5"   # used by "auto" / "drive"

# Only annotate files under this path prefix (relative to INPUT_FOLDER_ID).
# Set to None to consider every readable file in the tree.
# Example: "fineweb-edu-1b-smollm2-raw/"  -> shards/*.jsonl.gz, skips the tokenized sibling
INPUT_PATH_PREFIX = "fineweb-edu-1b-smollm2-raw/"

# Leave HF_PATH = None to use the hf_path found in the folder's meta.json.
HF_PATH = None            # e.g. "HuggingFaceFW/fineweb-edu"
HF_NAME = None            # config / subset, e.g. "sample-10BT"
HF_SPLIT = "train"
HF_TOKEN = None           # only needed for gated datasets

# ---- model (in your Drive) ----
MODEL_DIR = "/content/drive/MyDrive/co-lmlm-span-tagger/final"

# ---- output (My Drive, writable) ----
OUTPUT_DIR = Path("/content/drive/MyDrive/co-lmlm-annotations")

# ---- record fields (adjust to your data's schema) ----
TEXT_FIELD = "text"       # field/column holding the document text
ID_FIELD = "id"           # field/column holding a stable id (auto-generated if absent)
SOURCE_FIELD = "source"   # optional; falls back to SOURCE_DEFAULT
SOURCE_DEFAULT = "fineweb-edu"
INCLUDE_TEXT = True       # write the document text alongside its annotations (self-contained)

# ---- inference ----
MAX_LENGTH = 4096
BATCH = 32                # lower to 8-16 on a T4 if you hit OOM
ZSTD_LEVEL = 10

# ---- limits for a trial run (set to None for the full corpus) ----
MAX_FILES = 1             # Drive mode: input files per run
MAX_DOCS_PER_FILE = None  # Drive mode: docs per file
MAX_DOCS = 2000           # HF mode: docs this run; None streams the whole split
SHARD_DOCS = 50000        # HF mode: docs per output shard (also the resume unit)

LABELS = ["O", "B-FACT", "I-FACT"]
LABEL2ID = {n: i for i, n in enumerate(LABELS)}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output ->", OUTPUT_DIR)
print(f"input mode: {INPUT_MODE}")
print(f"path prefix: {INPUT_PATH_PREFIX!r}")
print(f"trial run: MAX_FILES={MAX_FILES}, MAX_DOCS={MAX_DOCS} (set to None for everything)")

## 3. Load the tuned model

Point `MODEL_DIR` at your checkpoint folder (the one containing `config.json`, `model.safetensors`,
and the tokenizer). If the default path is wrong, this cell searches your My Drive for a ModernBERT
token-classifier and reports what it finds.

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer


def find_model_dir(preferred):
    p = Path(preferred)
    if (p / "config.json").exists():
        return p
    print(f"{preferred} not found; searching My Drive for a ModernBERT tagger...")
    import json as _json
    for cfg in Path("/content/drive/MyDrive").rglob("config.json"):
        try:
            c = _json.loads(cfg.read_text())
        except Exception:
            continue
        if c.get("model_type") == "modernbert" and "B-FACT" in (c.get("label2id") or {}):
            print("  found:", cfg.parent)
            return cfg.parent
    raise FileNotFoundError("No ModernBERT tagger found — set MODEL_DIR to your checkpoint folder.")


model_dir = find_model_dir(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
         else torch.float16 if device == "cuda" else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = None
for attn in ("sdpa", "eager"):
    try:
        model = AutoModelForTokenClassification.from_pretrained(model_dir, attn_implementation=attn)
        break
    except Exception as exc:
        print(f"attn={attn} failed: {exc}")
if model is None:
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
if hasattr(model.config, "reference_compile"):
    model.config.reference_compile = False
model.to(device=device, dtype=dtype).eval()
print(f"loaded {model_dir}  on {device} ({dtype}), labels={model.config.id2label}")

### Span decoding

Turns the model's per-token BIO predictions into character-offset spans, trimming byte-level BPE's
leading-space so offsets land exactly on the fact text. Every returned span is a verbatim substring
of the source (`faithful` is true by construction).

In [ ]:
@torch.inference_mode()
def tag_batch(texts):
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True,
                    return_offsets_mapping=True, return_special_tokens_mask=True,
                    return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)
    preds = model(input_ids=input_ids, attention_mask=attn).logits.argmax(-1).cpu().tolist()
    offsets = enc["offset_mapping"].tolist()
    special = enc["special_tokens_mask"].tolist()
    out = []
    for b, text in enumerate(texts):
        spans, cur = [], None
        for k, tid in enumerate(preds[b]):
            s, e = offsets[b][k]
            if special[b][k] or e <= s:
                if cur:
                    spans.append(cur)
                    cur = None
                continue
            if tid == LABEL2ID["B-FACT"]:
                if cur:
                    spans.append(cur)
                cur = [s, e]
            elif tid == LABEL2ID["I-FACT"] and cur:
                cur[1] = e
            else:
                if cur:
                    spans.append(cur)
                    cur = None
        if cur:
            spans.append(cur)
        recs = []
        for s, e in spans:
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                recs.append({"span": text[s:e], "char_start": s, "char_end": e, "faithful": True})
        out.append(recs)
    return out


# quick sanity check
demo = "The Eiffel Tower was completed in 1889 and stands 330 metres tall in Paris."
print(tag_batch([demo])[0])

## 4. List the input files (Drive API)

Recursively lists every non-folder file in the shared input folder. If this errors with a
permission problem, confirm the folder is shared with the account you authenticated above.

In [ ]:
import io
import json

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

gdrive = build("drive", "v3")


def list_folder(folder_id):
    """Recursively list files. Each file gains a 'dir' key: its path below the root."""
    root = gdrive.files().get(fileId=folder_id, fields="name",
                              supportsAllDrives=True).execute()
    print(f"root folder: {root.get('name')!r} ({folder_id})")
    files = []
    stack = [(folder_id, "")]
    while stack:
        fid, prefix = stack.pop()
        token = None
        while True:
            resp = gdrive.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            for f in resp.get("files", []):
                if f["mimeType"] == "application/vnd.google-apps.folder":
                    stack.append((f["id"], f"{prefix}{f['name']}/"))
                else:
                    f["dir"] = prefix or "./"
                    files.append(f)
            token = resp.get("nextPageToken")
            if not token:
                break
    return files


def fetch_bytes(file_id):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(
        buf, gdrive.files().get_media(fileId=file_id, supportsAllDrives=True),
        chunksize=8 * 1024 * 1024,
    )
    ok = False
    while not ok:
        _, ok = dl.next_chunk()
    return buf.getvalue()


READABLE_SUFFIXES = (
    ".jsonl.zst", ".ndjson.zst", ".json.zst",
    ".jsonl.gz", ".ndjson.gz", ".json.gz",
    ".jsonl", ".ndjson", ".json",
    ".parquet", ".csv", ".tsv", ".txt",
)
# Most- to least-trustworthy source of an upstream HF repo id.
HF_PATH_KEYS = ("hf_path", "hf_dataset", "source_dataset", "dataset", "parent_corpus")
# `dataset`/`parent_corpus` often hold an internal id like "pretrain/fineweb-edu-10b",
# which looks like an HF repo id but is not one. Reject those prefixes.
NOT_HF_PREFIXES = ("pretrain/", "curriculum/", "sft/", "eval/", "probe/",
                   "vendor/", "tokenizer/")

input_files = []
HF_HINT = {}

if INPUT_MODE == "hf":
    print("INPUT_MODE='hf' — skipping the Drive folder")
else:
    all_files = list_folder(INPUT_FOLDER_ID)
    print(f"\n{len(all_files)} file(s) across "
          f"{len({f['dir'] for f in all_files})} folder(s):")
    shown = 0
    for d in sorted({f["dir"] for f in all_files}):
        in_dir = sorted((f for f in all_files if f["dir"] == d),
                        key=lambda f: -int(f.get("size") or 0))
        print(f"\n  {d}  ({len(in_dir)} file(s), "
              f"{sum(int(f.get('size') or 0) for f in in_dir)/1e6:.1f} MB)")
        for f in in_dir[:15]:
            mark = " " if f["name"].lower().endswith(READABLE_SUFFIXES) else "x"
            print(f"    {mark} {f['name']:48} {int(f.get('size') or 0)/1e6:8.2f} MB")
            shown += 1
        if len(in_dir) > 15:
            print(f"    ... and {len(in_dir) - 15} more")
    print('\n("x" = extension this notebook cannot read; e.g. pre-tokenized .bin shards)')

    # Small JSON sidecars describe the corpus. Read them: they name the upstream
    # dataset, which is where the raw text lives when the shards are tokenized.
    hints_by_key = {}
    for f in all_files:
        if not f["name"].lower().endswith(".json") or int(f.get("size") or 0) > 2_000_000:
            continue
        try:
            obj = json.loads(fetch_bytes(f["id"]).decode("utf-8"))
        except Exception as e:
            print(f"\n{f['dir']}{f['name']}: unreadable ({type(e).__name__})")
            continue
        if not isinstance(obj, dict):
            continue
        print(f"\n{f['dir']}{f['name']}: {json.dumps(obj, indent=2)[:800]}")
        for k in HF_PATH_KEYS:
            v = obj.get(k)
            if (isinstance(v, str) and v.count("/") == 1
                    and not v.startswith(NOT_HF_PREFIXES)):
                hints_by_key.setdefault(k, v)
        if isinstance(obj.get("hf_name"), str) and "/" not in obj["hf_name"]:
            HF_HINT.setdefault("name", obj["hf_name"])
    for k in HF_PATH_KEYS:
        if k in hints_by_key:
            HF_HINT["path"] = hints_by_key[k]
            break

    # Largest first, so a trial run hits a real shard rather than a metadata file.
    def _under_prefix(f):
        if not INPUT_PATH_PREFIX:
            return True
        rel = f"{f.get('dir', '')}{f['name']}"
        return rel.replace("\\", "/").startswith(INPUT_PATH_PREFIX)

    input_files = sorted(
        (f for f in all_files
         if f["name"].lower().endswith(READABLE_SUFFIXES) and _under_prefix(f)),
        key=lambda f: int(f.get("size") or 0),
        reverse=True,
    )
    total_mb = sum(int(f.get("size") or 0) for f in input_files) / 1e6
    print(f"\n{len(input_files)} readable candidate(s)"
          + (f" under {INPUT_PATH_PREFIX!r}" if INPUT_PATH_PREFIX else "")
          + f", {total_mb:.1f} MB total")

if HF_HINT:
    print(f"\nupstream dataset detected in metadata: {HF_HINT}")
print(f"HF fallback -> path={HF_PATH or HF_HINT.get('path')!r}, "
      f"name={HF_NAME or HF_HINT.get('name')!r}, split={HF_SPLIT!r}")

## 5. Annotate (resumable)

Downloads each file, reads its records (JSONL / JSON / Parquet / CSV / TXT, optionally `.zst` or `.gz`),
tags every document, and writes an output shard next to a manifest. Re-running skips finished files.

In [ ]:
import gzip
import io
import json
import time

import zstandard
from googleapiclient.http import MediaIoBaseDownload

TMP = Path("/content/_annotate_tmp")
TMP.mkdir(exist_ok=True)
MANIFEST = OUTPUT_DIR / "_manifest.json"


def load_manifest():
    if not MANIFEST.exists():
        return {"files": [], "hf": {}}
    raw = json.loads(MANIFEST.read_text())
    if isinstance(raw, list):  # older format: a bare list of finished file names
        return {"files": raw, "hf": {}}
    raw.setdefault("files", [])
    raw.setdefault("hf", {})
    return raw


def save_manifest(m):
    MANIFEST.write_text(json.dumps(m, indent=2, sort_keys=True))


def download(file_id, dest):
    req = gdrive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024)
        ok = False
        while not ok:
            _, ok = dl.next_chunk()


def _iter_jsonl(fh):
    for line in fh:
        line = line.strip()
        if line:
            yield json.loads(line)


def read_records(path, name):
    low = name.lower()
    if low.endswith(".zst"):
        with open(path, "rb") as raw, zstandard.ZstdDecompressor().stream_reader(raw) as r:
            text = io.TextIOWrapper(r, encoding="utf-8")
            if low.endswith((".jsonl.zst", ".ndjson.zst")):
                yield from _iter_jsonl(text)
            elif low.endswith(".json.zst"):
                obj = json.load(text)
                yield from (obj if isinstance(obj, list) else obj.get("data", [obj]))
            else:
                raise ValueError(f"unsupported compressed type: {name}")
    elif low.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            if low.endswith((".jsonl.gz", ".ndjson.gz")):
                yield from _iter_jsonl(fh)
            elif low.endswith(".json.gz"):
                obj = json.load(fh)
                yield from (obj if isinstance(obj, list) else obj.get("data", [obj]))
            else:
                raise ValueError(f"unsupported compressed type: {name}")
    elif low.endswith((".jsonl", ".ndjson")):
        with open(path, encoding="utf-8") as fh:
            yield from _iter_jsonl(fh)
    elif low.endswith(".json"):
        obj = json.loads(Path(path).read_text(encoding="utf-8"))
        yield from (obj if isinstance(obj, list) else obj.get("data", [obj]))
    elif low.endswith(".parquet"):
        import pyarrow.parquet as pq
        for batch in pq.ParquetFile(path).iter_batches(batch_size=1024):
            for row in batch.to_pylist():
                yield row
    elif low.endswith((".csv", ".tsv")):
        import csv
        with open(path, encoding="utf-8", newline="") as fh:
            reader = csv.DictReader(fh, delimiter="\t" if low.endswith(".tsv") else ",")
            yield from reader
    elif low.endswith(".txt"):
        yield {TEXT_FIELD: Path(path).read_text(encoding="utf-8")}
    else:
        raise ValueError(f"unsupported file type: {name}")


def has_documents(path, name, probe=5):
    """True if the first records carry text under TEXT_FIELD.

    Distinguishes corpus shards from readable-but-empty metadata (meta.json,
    subsets.json, manifest.json) without hard-coding file names.
    """
    keys = []
    try:
        for i, rec in enumerate(read_records(path, name)):
            if isinstance(rec, dict):
                keys = sorted(rec.keys())
                val = rec.get(TEXT_FIELD)
                if isinstance(val, str) and val.strip():
                    return True, keys
            if i + 1 >= probe:
                break
    except Exception as e:
        return False, [f"unreadable: {type(e).__name__}: {e}"]
    return False, keys


class ShardWriter:
    """Writes annotation shards, creating a file only once there is a record for it."""

    def __init__(self, stem, shard_docs=None, start_index=0):
        self.stem = stem
        self.shard_docs = shard_docs
        self.index = start_index
        self.paths = []
        self._fout = self._w = self._path = None
        self._in_shard = 0

    def _open(self):
        suffix = "" if self.shard_docs is None else f"-{self.index:05d}"
        self._path = OUTPUT_DIR / f"{self.stem}{suffix}.annotations.jsonl.zst"
        self._fout = open(self._path, "wb")
        self._w = zstandard.ZstdCompressor(level=ZSTD_LEVEL).stream_writer(self._fout)
        self._in_shard = 0

    def write(self, rec):
        if self._w is None:
            self._open()
        self._w.write((json.dumps(rec, ensure_ascii=False) + "\n").encode("utf-8"))
        self._in_shard += 1
        if self.shard_docs and self._in_shard >= self.shard_docs:
            self.close_shard()

    def close_shard(self):
        if self._w is None:
            return None
        self._w.close()
        self._fout.close()
        self.paths.append(self._path)
        finished, self._path = self._path, None
        self._w = self._fout = None
        self.index += 1
        self._in_shard = 0
        return finished


def annotate_documents(records, stem, max_docs=None, shard_docs=None, start_index=0,
                       label="", on_shard=None):
    """Tag every record carrying TEXT_FIELD; return (paths, n_docs, n_spans)."""
    writer = ShardWriter(stem, shard_docs, start_index)
    n_docs = n_spans = 0
    texts, metas = [], []
    t0 = time.time()

    def flush():
        nonlocal n_docs, n_spans
        if not texts:
            return
        for meta, spans in zip(metas, tag_batch(texts)):
            rec = {"id": meta["id"], "source": meta["source"], "annotations": spans}
            if INCLUDE_TEXT:
                rec["text"] = meta["text"]
            before = writer.index
            writer.write(rec)
            if on_shard is not None and writer.index != before:
                on_shard(writer.index)
            n_docs += 1
            n_spans += len(spans)
        texts.clear()
        metas.clear()

    for n, rec in enumerate(records):
        if max_docs is not None and n_docs + len(texts) >= max_docs:
            break
        if MAX_DOCS_PER_FILE and n >= MAX_DOCS_PER_FILE:
            break
        text = rec.get(TEXT_FIELD) if isinstance(rec, dict) else None
        if not text or not isinstance(text, str):
            continue
        texts.append(text[: MAX_LENGTH * 12])  # generous char cap; tokenizer truncates anyway
        metas.append({
            "id": str(rec.get(ID_FIELD) or f"{stem}-{n}"),
            "source": rec.get(SOURCE_FIELD, SOURCE_DEFAULT),
            "text": text,
        })
        if len(texts) >= BATCH:
            flush()
            if n_docs % (BATCH * 20) == 0:
                rate = n_docs / max(time.time() - t0, 1e-9)
                print(f"    {label}: {n_docs:,} docs, {n_spans:,} spans, {rate:.0f} docs/s")
    flush()
    last = writer.close_shard()
    if last is not None and on_shard is not None:
        on_shard(writer.index)
    return writer.paths, n_docs, n_spans


def annotate_drive_file(f):
    name = f["name"]
    stem = name
    for suffix in (".gz", ".zst", ".jsonl", ".ndjson", ".json", ".parquet", ".csv", ".tsv", ".txt"):
        if stem.lower().endswith(suffix):
            stem = stem[: -len(suffix)]
    local = TMP / name
    download(f["id"], local)
    ok, keys = has_documents(local, name)
    if not ok:
        local.unlink(missing_ok=True)
        return None, 0, 0, keys
    paths, nd, ns = annotate_documents(read_records(local, name), stem, label=name)
    local.unlink(missing_ok=True)
    return (paths[0] if paths else None), nd, ns, keys


def annotate_hf(path, name, split, manifest):
    from datasets import load_dataset

    key = f"{path}|{name or ''}|{split}"
    done_shards = int(manifest["hf"].get(key, 0))
    skip = done_shards * SHARD_DOCS
    print(f"streaming {path}" + (f" [{name}]" if name else "") + f" split={split}")
    if skip:
        print(f"  resuming after {done_shards} shard(s) = {skip:,} docs")
    ds = load_dataset(path, name, split=split, streaming=True, token=HF_TOKEN)
    if skip:
        ds = ds.skip(skip)

    def record_shard(index):
        manifest["hf"][key] = index
        save_manifest(manifest)

    stem = path.replace("/", "__") + (f"-{name}" if name else "") + f"-{split}"
    return annotate_documents(
        ds, stem, max_docs=MAX_DOCS, shard_docs=SHARD_DOCS,
        start_index=done_shards, label=stem, on_shard=record_shard,
    )


# ---- run ----
manifest = load_manifest()
done = set(manifest["files"])
grand_docs = grand_spans = 0
annotated = skipped_meta = 0
start = time.time()

if INPUT_MODE in ("auto", "drive"):
    candidates = [f for f in input_files if f["name"] not in done]
    print(f"{len(candidates)} Drive candidate(s) ({len(done)} already done)\n")
    for f in candidates:
        if MAX_FILES and annotated >= MAX_FILES:
            break
        print(f"[{annotated + 1}] {f['name']} ({int(f.get('size') or 0)/1e6:.1f} MB)")
        out_path, nd, ns, keys = annotate_drive_file(f)
        if out_path is None:
            skipped_meta += 1
            print(f"    skipped: no {TEXT_FIELD!r} in records; keys seen: {keys}")
            continue
        annotated += 1
        grand_docs += nd
        grand_spans += ns
        manifest["files"] = sorted(done | {f["name"]})
        done = set(manifest["files"])
        save_manifest(manifest)
        print(f"    -> {out_path.name}: {nd:,} docs, {ns:,} spans "
              f"({ns/max(nd,1):.1f}/doc)  [{(time.time()-start)/60:.1f} min elapsed]")

if annotated == 0 and INPUT_MODE in ("auto", "hf"):
    hf_path = HF_PATH or HF_HINT.get("path")
    hf_name = HF_NAME or HF_HINT.get("name")
    if not hf_path:
        raise SystemExit(
            "no text documents in Drive and no HF dataset to fall back to — "
            "set HF_PATH (e.g. 'HuggingFaceFW/fineweb-edu') in the config cell"
        )
    if skipped_meta:
        print(f"\nDrive folder held no documents ({skipped_meta} metadata file(s)); "
              "falling back to the upstream dataset\n")
    paths, nd, ns = annotate_hf(hf_path, hf_name, HF_SPLIT, manifest)
    grand_docs += nd
    grand_spans += ns
    for p in paths:
        print(f"    -> {p.name}")

print(f"\ndone: {grand_docs:,} docs, {grand_spans:,} spans "
      f"[{(time.time()-start)/60:.1f} min]")
if grand_docs:
    print(f"output in {OUTPUT_DIR}")
else:
    print(f"nothing annotated — no record carried TEXT_FIELD={TEXT_FIELD!r}.")
    print("  - if the 'keys seen' above name a different text column, set TEXT_FIELD to it")
    print("  - if the folder holds only .bin token shards + sidecars, there is no raw text "
          "there: set INPUT_MODE='hf' and HF_PATH to the upstream corpus")
    print(f"  - otherwise INPUT_FOLDER_ID ({INPUT_FOLDER_ID}) is the wrong folder")

## 6. Verify the output

Reads back the first few annotated records from the newest shard and checks that every span is a
faithful substring at its recorded offset.

In [ ]:
import zstandard

shards = sorted(OUTPUT_DIR.glob("*.annotations.jsonl.zst"))
print(f"{len(shards)} output shard(s):")
for s in shards:
    print(f"  {s.name}  {s.stat().st_size/1e6:.2f} MB")

# Prefer a non-empty shard (empty ones come from sidecar files / wrong TEXT_FIELD).
candidates = [s for s in reversed(shards) if s.stat().st_size > 0]
if not candidates:
    print("\nno non-empty shards to verify — delete empty *.annotations.jsonl.zst, "
          "clear _manifest.json, and re-run annotate on a real corpus file")
else:
    shard = candidates[0]
    print(f"\nverifying {shard.name}")
    with open(shard, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as r:
        text = io.TextIOWrapper(r, encoding="utf-8")
        bad = 0
        n = -1  # stays -1 if the shard has no lines
        for n, line in enumerate(text):
            rec = json.loads(line)
            if "text" in rec:
                for a in rec["annotations"]:
                    if rec["text"][a["char_start"]:a["char_end"]] != a["span"]:
                        bad += 1
            if n < 2:
                print(f"\n{rec['id']}  ({len(rec['annotations'])} spans)")
                for a in rec["annotations"][:8]:
                    print(f"  [{a['char_start']:>5}] {a['span']!r}")
            if n >= 2000:
                break
        print(f"\nchecked {n + 1} records, {bad} offset mismatches")